In [ ]:
# --- repo root + config (walk parents; do not use ../..) ---
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



In [ ]:
# === Setup (Part 2) — keep byte-identical across RQ notebooks ===
import json
import os
import pickle
from pathlib import Path

import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

CONFIG_PATH = (PROJECT_ROOT / "config" / "config.json")
with open(CONFIG_PATH, "r", encoding="utf-8") as _f:
    CFG = json.load(_f)

PROJECT_ROOT = CONFIG_PATH.parent


def _expand_tree(obj):
    """Expand ~ in all string paths; leave non-strings / null unchanged."""
    if isinstance(obj, dict):
        return {k: _expand_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_expand_tree(v) for v in obj]
    if isinstance(obj, str):
        return os.path.expanduser(obj)
    return obj


CFG = _expand_tree(CFG)


def _resolve_cfg_path(p):
    if p is None:
        return None
    path = Path(p)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path


def _model_src(key: str) -> str:
    """Return local path or HF hub id for a model key in CFG['models']."""
    if key not in CFG["models"]:
        raise KeyError(f"Unknown model key {key!r}. Choose from: {sorted(CFG['models'])}")
    return CFG["models"][key]


def load_generative(key: str):
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        src,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    return tokenizer, model


def load_encoder(key: str):
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src)
    model = AutoModel.from_pretrained(src)
    return tokenizer, model


def free_model(model):
    del model
    torch.cuda.empty_cache()


_pool_path = _resolve_cfg_path(CFG["pool_full"])
with open(_pool_path, "rb") as _f:
    cui_pool = pickle.load(_f)

_n_cuis = cui_pool.get("n_cuis", len(cui_pool.get("cuis", {})))
print(f"Config: {CONFIG_PATH}")
print(f"Loaded CUI pool from: {_pool_path}")
print(f"Pool type: {cui_pool.get('pool_type', 'unknown')} | unique CUIs: {_n_cuis:,}")
if _n_cuis < 10_000:
    print("WARNING: CUI count looks like the MeSH subset (~1,201), not full UMLS.")
else:
    print("Confirmed: full_umls-scale CUI pool loaded.")


# RQ2: Accuracy-Stability Dissociation in Clinical LLMs
## SIT723 - masters Research Techniques and Applications

**Research Question:**
Does semantic entropy detect systematic interpretation instability in
high-accuracy LLM outputs that traditional evaluation metrics fail to
reveal, and does this accuracy-stability dissociation hold across both
encoder-only and generative model architectures?

**Gaps addressed (Task 2.3D):**
- **Gap 1:** Overreliance on benchmark accuracy despite evidence
  that accuracy hides instability under meaning-preserving variation
  (Sclar et al., 2024; Agrawal et al., 2023; Hager et al., 2024).
  This notebook tests whether high-accuracy models simultaneously
  exhibit elevated semantic entropy - demonstrating that accuracy-based
  evaluation underestimates reliability risk.
- **Gap 4:** Absence of a structured, ontology-grounded metric for
  evaluating the illusion of understanding in clinical language settings.
  UMLS-grounded entropy is proposed as that metric.

**Sub-questions:**
- **SQ1:** Can a model achieve high task accuracy while still producing
  inconsistent semantic outputs in the high-accuracy quartile?
- **SQ2:** Does the dissociation appear in both encoder-only (CUI-ranking
  entropy) and generative (text-generation entropy) models, or is it
  architecture-specific?
- **SQ3:** How often do apparently correct outputs remain semantically
  unstable when measured through UMLS-grounded entropy?

**Statistical operationalisation:**
- Primary: One-tailed Wilcoxon signed-rank test, top vs bottom accuracy
  quartile. Pre-specified threshold: rank-biserial r >= 0.30
- Global: Spearman rho between per-instance accuracy and H-hat.
  Pre-specified threshold: |rho| < 0.30 (dissociation confirmed if weak)
- BH-FDR correction at q = 0.05

**Pipeline:**
MedMentions ST21pv · N=550 · k=8 perturbations · Six-gate validation
Encoder models: BERT-base, BioBERT, PubMedBERT (CUI-ranking entropy)
Generative models: FLAN-T5-base (250M), FLAN-T5-XXL (11B INT8),
BioMistral-7B (INT8) (text-generation entropy)

**Scope note:** Perturbation generation and gate validation are shared
with RQ1. This notebook loads the validated perturbations from RQ1
intermediate outputs and adds generative model inference on top.

## 1) Environment Setup
Installs and imports all dependencies. Sets random seed and detects GPU.
This notebook reuses validated perturbations from RQ1 - no perturbation
generation needed here.

In [ ]:
import os, sys, math, json, random, warnings, gc, time
import subprocess
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import torch
from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForSeq2SeqLM, AutoModelForCausalLM,
    T5ForConditionalGeneration, T5Tokenizer,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
              f"Free: {free/1e9:.1f}GB / {total/1e9:.1f}GB")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

# Config
PILOT_N        = 550      # must match RQ1 run
# Encoder models - loaded from RQ1 outputs (pre-run in RQ1 notebook)
ENCODER_MODELS = ["BERT-base", "BioBERT", "PubMedBERT"]
# Encoder checkpoints used in RQ1 (for traceability)
ENCODER_CHECKPOINTS = {
    "BERT-base":   "bert-base-uncased",
    "BioBERT":     "dmis-lab/biobert-v1.1",
    "PubMedBERT":  "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
}
K_PERTURB      = 8        # must match RQ1 run
BOOTSTRAP_B    = 1000
TOP_Q          = 0.75     # top accuracy quartile threshold
BOT_Q          = 0.25     # bottom accuracy quartile threshold
# INT8 quantisation for large models (saves ~50% VRAM)
USE_INT8       = True
# Greedy decoding - eliminates sampling stochasticity as confound
DO_SAMPLE      = False
TEMPERATURE    = 1.0
MAX_NEW_TOKENS = 64

# Paths - must match RQ1 output structure
PROJECT_ROOT   = Path("/home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs")
RQ1_INTER      = PROJECT_ROOT / "outputs" / "rq1" / "intermediate"
OUTPUT_DIR     = PROJECT_ROOT / "outputs" / "rq2"
FIGURES_DIR    = OUTPUT_DIR / "figures"
TABLES_DIR     = OUTPUT_DIR / "tables"
for p in [OUTPUT_DIR, FIGURES_DIR, TABLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"\nConfig: PILOT_N={PILOT_N} | K_PERTURB={K_PERTURB} | "
      f"USE_INT8={USE_INT8} | BOOTSTRAP_B={BOOTSTRAP_B}")

## 2) Load RQ1 Validated Perturbations and Encoder Results
RQ2 reuses the validated perturbations and encoder entropy scores
already computed in RQ1. This avoids re-running the expensive
perturbation generation and six-gate validation pipeline.

**What is loaded:**
- `rq1_sampled_instances.csv` - the 550 sampled MedMentions instances
- `rq1_linguistic_features.csv` - accepted perturbations with linguistic features
- `rq1_conditional_entropy_scores.csv` - encoder entropy scores (H-hat)
  from BERT-base, BioBERT, PubMedBERT (CUI-ranking entropy)
- `rq1_model_outputs.csv` - per-perturbation CUI predictions and accuracy

In [ ]:
# Load RQ1 validated perturbations
df_instances = pd.read_csv(RQ1_INTER / "rq1_sampled_instances.csv")
df_features  = pd.read_csv(RQ1_INTER / "rq1_linguistic_features.csv")
df_encoder   = pd.read_csv(RQ1_INTER / "rq1_model_outputs.csv")

# Load RQ1 conditional entropy scores
rq1_entropy_path = PROJECT_ROOT / "outputs" / "rq1" / "tables" / \
                   "rq1_conditional_entropy_scores.csv"
if rq1_entropy_path.exists():
    df_enc_entropy = pd.read_csv(rq1_entropy_path)
else:
    # fallback: recompute from model outputs
    df_enc_entropy = None
    print("[WARN] Conditional entropy CSV not found - will recompute")

print(f"Instances loaded:      {len(df_instances)}")
print(f"Accepted perturbations: {len(df_features)}")
print(f"Encoder model outputs:  {len(df_encoder)}")
print(f"Models in encoder data: {df_encoder['model_name'].unique().tolist()}")
print(f"\nAccepted by type:")
print(df_features["perturbation_type"].value_counts().to_string())

# Build accepted_df for generative inference
accepted_df = df_features.copy()
print(f"\nSample context:")
print(accepted_df[["instance_id","perturbation_type",
                    "mention_context","linguistic_category"]].head(3).to_string())

## 3) Encoder Entropy (CUI-Ranking) - From RQ1

The encoder entropy scores from BERT-base, BioBERT, and PubMedBERT
were already computed in RQ1 using UMLS-KB cosine similarity CUI
assignment. This section prepares those scores for RQ2 analysis.

**CUI-ranking entropy:** For each instance, the encoder model assigns
a predicted CUI to each perturbation. Entropy is computed over the
distribution of predicted CUIs. High entropy = model assigns different
concepts to meaning-equivalent inputs = semantic instability.

In [ ]:
def compute_normalised_entropy(predictions: list) -> float:
    """Shannon entropy over prediction distribution, normalised by log2(k+1)."""
    k = len(predictions)
    if k <= 1:
        return 0.0
    counts = Counter(predictions)
    probs  = [c / k for c in counts.values()]
    H      = -sum(p * math.log2(p) for p in probs if p > 0)
    H_max  = math.log2(k + 1)
    return H / H_max if H_max > 0 else 0.0

# Compute per-instance entropy from RQ1 encoder predictions
encoder_rows = []
for (iid, model), grp in df_encoder.groupby(
        ["instance_id", "model_name"]):
    perts = grp[grp["input_type"] == "perturbation"]
    if len(perts) == 0:
        continue
    preds    = perts["predicted_cui_or_cluster"].tolist()
    H_hat    = compute_normalised_entropy(preds)
    accuracy = perts["accuracy_correct"].mean()
    encoder_rows.append({
        "instance_id":           iid,
        "model_name":            model,
        "architecture":          "encoder",
        "normalised_entropy":    H_hat,
        "mean_accuracy":         accuracy,
        "n_perturbations":       len(perts),
    })

df_enc_rq2 = pd.DataFrame(encoder_rows)
print(f"Encoder entropy rows: {len(df_enc_rq2)}")
print("\nEncoder entropy summary by model:")
print(df_enc_rq2.groupby("model_name")[
    ["normalised_entropy","mean_accuracy"]
].agg(["mean","std"]).round(4).to_string())

## 4) Generative Model Inference (Text-Generation Entropy)

Three generative models are evaluated at two parameter scales:

| Model | Parameters | Quantisation | Purpose |
|---|---|---|---|
| FLAN-T5-base | 250M | FP32 | Small-scale generative baseline |
| FLAN-T5-XXL | 11B | INT8 | Large-scale generative (scale comparison) |
| BioMistral-7B | 7B | INT8 | Biomedical generative (domain adaptation) |

**Text-generation entropy:** For each perturbation, the generative
model produces a text answer. Shannon entropy is computed over the
distribution of generated answers across perturbations using
semantic equivalence clustering (exact-match + UMLS normalisation).

**Greedy decoding (T=0):** All models use greedy decoding to eliminate
sampling stochasticity as a confound. Any entropy observed is due to
sensitivity to input variation, not random sampling.

In [ ]:
import csv as _csv

# Generative model registry
GEN_MODEL_SPECS = {
    "FLAN-T5-base": {
        "hf_id":    "google/flan-t5-base",
        "type":     "seq2seq",
        "int8":     False,   # small enough for FP32
    },
    "FLAN-T5-XXL": {
        "hf_id":    "google/flan-t5-xxl",
        "type":     "seq2seq",
        "int8":     USE_INT8,
    },
    "BioMistral-7B": {
        "hf_id":    "BioMistral/BioMistral-7B",
        "type":     "causal",
        "int8":     USE_INT8,
    },
}

def load_gen_model(model_name: str, spec: dict):
    """Load a generative model with optional INT8 quantisation."""
    hf_id  = spec["hf_id"]
    use_i8 = spec["int8"]
    mtype  = spec["type"]

    print(f"Loading {model_name} ({hf_id}) ...")

    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # INT8 config - only for large models that need it
    bnb_config = None
    if use_i8 and DEVICE == "cuda":
        try:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(
                load_in_8bit=True,
                llm_int8_enable_fp32_cpu_offload=True,
            )
            print(f"  Using INT8 quantisation")
        except ImportError:
            print("  [WARN] bitsandbytes not available - loading FP16")

    if bnb_config is not None:
        # INT8 large models: use device_map="auto" for multi-GPU
        load_kwargs = dict(
            quantization_config=bnb_config,
            device_map="auto",
        )
    elif DEVICE == "cuda":
        # Small models (FP16): single GPU, no device_map
        load_kwargs = dict(
            torch_dtype=torch.float16,
            device_map=None,
        )
    else:
        # CPU fallback
        load_kwargs = dict(device_map=None)

    if mtype == "seq2seq":
        model = T5ForConditionalGeneration.from_pretrained(
            hf_id, **load_kwargs, weights_only=False)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            hf_id, **load_kwargs, weights_only=False)

    # Only call .to() when NOT using device_map
    if bnb_config is None and DEVICE == "cuda":
        model = model.to(DEVICE)

    model.eval()

    if DEVICE == "cuda":
        free, total = torch.cuda.mem_get_info(0)
        print(f"  GPU free after load: {free/1e9:.1f}GB / "
              f"{total/1e9:.1f}GB")
    return tokenizer, model


def generate_answer(text: str, mention: str, tokenizer, model,
                    model_type: str) -> str:
    """
    Generate a clinical concept answer for a given input text.
    Uses greedy decoding (do_sample=False) to eliminate sampling noise.
    """
    if model_type == "seq2seq":
        prompt = (f"What is the primary medical concept mentioned in this text? "
                  f"Answer with the concept name only.\n\nText: {text}\n\n"
                  f"Medical concept:")
    else:
        prompt = (f"<s>[INST] Identify the primary medical concept in the following "
                  f"clinical text. Reply with only the concept name.\n\n"
                  f"Text: {text} [/INST]")

    enc = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=512,
                    padding=True)
    # device_map="auto" may split model across multiple GPUs.
    # Route inputs to the device of the model's first parameter
    # rather than hardcoding cuda:0.
    try:
        _first_device = next(model.parameters()).device
        enc = {k: v.to(_first_device) for k, v in enc.items()}
    except StopIteration:
        enc = {k: v.to("cuda:0") for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            pad_token_id=tokenizer.pad_token_id,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    # For causal models, remove the prompt from the output
    if model_type == "causal" and prompt in decoded:
        decoded = decoded.replace(prompt, "").strip()
    return decoded[:200]  # cap output length


# Run generative inference
gen_output_path = TABLES_DIR / "rq2_generative_outputs.csv"
gen_rows = []

# Build variants list (original + accepted perturbations)
orig_rows = []
for _, r in df_instances.iterrows():
    orig_rows.append({
        "instance_id":    r["instance_id"],
        "input_type":     "original",
        "perturbation_type": "original",
        "input_text":     r["mention_context"],
        "gold_mention":   r.get("gold_mention", ""),
        "gold_cui":       r.get("gold_cui", ""),
    })

pert_rows_gen = []
for _, r in accepted_df.iterrows():
    pert_rows_gen.append({
        "instance_id":    r["instance_id"],
        "input_type":     "perturbation",
        "perturbation_type": r["perturbation_type"],
        "input_text":     r["perturbation_text"],
        "gold_mention":   r.get("gold_mention", ""),
        "gold_cui":       r.get("gold_cui", ""),
    })

variants_gen = pd.DataFrame(orig_rows + pert_rows_gen)
print(f"Total generative inference variants: {len(variants_gen)}")

for model_name, spec in GEN_MODEL_SPECS.items():
    print(f"\n{'='*60}")
    print(f"Running generative inference: {model_name}")
    tokenizer, model = load_gen_model(model_name, spec)

    for _, v in variants_gen.iterrows():
        answer = generate_answer(
            v["input_text"], v["gold_mention"],
            tokenizer, model, spec["type"]
        )
        gen_rows.append({
            "instance_id":       v["instance_id"],
            "model_name":        model_name,
            "architecture":      "generative",
            "input_type":        v["input_type"],
            "perturbation_type": v["perturbation_type"],
            "input_text":        v["input_text"],
            "gold_mention":      v["gold_mention"],
            "gold_cui":          v["gold_cui"],
            "generated_answer":  answer,
            "mapping_mode":      "text-generation",
        })

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        print(f"GPU cleared after {model_name}")

df_gen = pd.DataFrame(gen_rows)
df_gen.to_csv(gen_output_path, index=False,
              quoting=_csv.QUOTE_MINIMAL, escapechar="\\")
print(f"\nSaved generative outputs: {len(df_gen)} rows → {gen_output_path}")
print(df_gen.groupby("model_name").size().to_string())

## 5) Text-Generation Entropy Computation

For each generative model, semantic entropy is computed over the
distribution of generated answers across perturbations.

**Semantic equivalence clustering:**
Two answers are treated as equivalent if:
1. Exact string match (case-insensitive, whitespace-normalised), OR
2. Both resolve to the same UMLS concept via string lookup against
   the candidate pool built in RQ1

**Accuracy operationalisation for generative models:**
An answer is considered correct if the generated text contains the
gold mention (or a known UMLS synonym) as a substring.

In [ ]:
# Load UMLS pool for synonym matching
umls_pool_path = RQ1_INTER / "umls_candidate_pool.csv"
if umls_pool_path.exists():
    df_umls = pd.read_csv(umls_pool_path)
    # Build mention -> set of synonyms lookup
    syn_lookup = defaultdict(set)
    for _, r in df_umls.iterrows():
        syn_lookup[r["source_mention"].lower()].add(
            r["candidate_text"].lower()
        )
    print(f"Loaded UMLS synonym pool: {len(df_umls)} entries")
else:
    syn_lookup = defaultdict(set)
    print("[WARN] UMLS pool not found - using exact match only")


def normalise_answer(text: str) -> str:
    """Normalise generated answer for semantic equivalence comparison."""
    return " ".join(str(text).lower().strip().split())


def answer_correct(generated: str, gold_mention: str) -> int:
    """Check if generated answer contains gold mention or UMLS synonym."""
    gen = normalise_answer(generated)
    gm  = normalise_answer(gold_mention)
    if gm in gen:
        return 1
    for syn in syn_lookup.get(gm, set()):
        if syn in gen:
            return 1
    return 0


def cluster_answers(answers: list) -> list:
    """
    Cluster answers by semantic equivalence.
    Returns a list of cluster labels (one per answer).
    """
    normalised = [normalise_answer(a) for a in answers]
    # Build equivalence classes: answers that normalise to the same string
    # are in the same cluster
    seen = {}
    labels = []
    for norm in normalised:
        if norm not in seen:
            seen[norm] = len(seen)
        labels.append(seen[norm])
    return labels


gen_entropy_rows = []

for (iid, model_name), grp in df_gen.groupby(
        ["instance_id", "model_name"]):
    perts = grp[grp["input_type"] == "perturbation"]
    if len(perts) == 0:
        continue

    answers  = perts["generated_answer"].tolist()
    clusters = cluster_answers(answers)
    H_hat    = compute_normalised_entropy(clusters)

    # Accuracy: fraction of perturbation answers containing gold mention
    gold_mention = perts["gold_mention"].iloc[0]
    accuracy = np.mean([
        answer_correct(a, gold_mention) for a in answers
    ])

    gen_entropy_rows.append({
        "instance_id":        iid,
        "model_name":         model_name,
        "architecture":       "generative",
        "normalised_entropy": H_hat,
        "mean_accuracy":      accuracy,
        "n_perturbations":    len(perts),
        "n_unique_answers":   len(set(normalise_answer(a) for a in answers)),
    })

df_gen_entropy = pd.DataFrame(gen_entropy_rows)
print(f"Generative entropy rows: {len(df_gen_entropy)}")
print("\nGenerative entropy summary by model:")
print(df_gen_entropy.groupby("model_name")[
    ["normalised_entropy","mean_accuracy","n_unique_answers"]
].agg(["mean","std"]).round(4).to_string())

## 6) Combine Encoder and Generative Entropy

Combines encoder (CUI-ranking) and generative (text-generation)
entropy into a single analysis DataFrame for RQ2.

In [ ]:
df_all = pd.concat(
    [df_enc_rq2, df_gen_entropy],
    ignore_index=True,
    sort=False
)

print(f"Combined rows: {len(df_all)}")
print(f"\nAll models:")
print(df_all.groupby(["model_name","architecture"])[
    ["normalised_entropy","mean_accuracy"]
].agg(["mean","std"]).round(4).to_string())

# Save combined table
df_all.to_csv(TABLES_DIR / "rq2_all_entropy.csv", index=False)
print(f"\nSaved: {TABLES_DIR}/rq2_all_entropy.csv")

## 6b) Negative Controls

Two negative control conditions verify the pipeline behaves correctly:

**Control 1 - Identical copy:** Four identical copies of each
original input fed to every model. Entropy MUST equal 0.
Any model with mean H > 0.05 on identical inputs is flagged
as a determinism failure and excluded from RQ2 analysis.

**Control 2 - Typographic variant:** Upper-cased, whitespace-
normalised version of each input. Entropy MUST be near 0 (H ≈ 0).

Both controls were pre-specified in the RQ2 methodology (Milestone M5).

In [ ]:
# ── Negative controls (Milestone M5) ─────────────────────────────────────
EXCLUDED_MODELS_RQ2 = set()
nc_rows = []

# Sample 30 instances for control testing
nc_sample = df_instances.head(30)

for model_name, spec in GEN_MODEL_SPECS.items():
    print(f"Running negative controls: {model_name}")
    tokenizer, model = load_gen_model(model_name, spec)

    id_Hs, ty_Hs = [], []
    for _, inst in nc_sample.iterrows():
        ctx = inst["mention_context"]

        # Control 1: identical copy (k=4)
        id_answers = [
            generate_answer(ctx, inst.get("gold_mention",""),
                            tokenizer, model, spec["type"])
            for _ in range(4)
        ]
        id_clusters = cluster_answers(id_answers)
        id_Hs.append(compute_normalised_entropy(id_clusters))

        # Control 2: typographic variant
        typo = ctx.upper().replace("  "," ").strip()
        ty_answers = [
            generate_answer(typo, inst.get("gold_mention",""),
                            tokenizer, model, spec["type"])
            for _ in range(4)
        ]
        ty_clusters = cluster_answers(ty_answers)
        ty_Hs.append(compute_normalised_entropy(ty_clusters))

    mean_id = float(np.mean(id_Hs))
    mean_ty = float(np.mean(ty_Hs))
    flag    = mean_id > 0.05

    if flag:
        EXCLUDED_MODELS_RQ2.add(model_name)
        print(f"  ⚠️  EXCLUDING {model_name}: "
              f"H(identical)={mean_id:.4f} > 0.05")
    else:
        print(f"  ✓ {model_name}: H(identical)={mean_id:.4f}, "
              f"H(typo)={mean_ty:.4f}")

    nc_rows.append({
        "model_name":       model_name,
        "architecture":     "generative",
        "identical_copy_H": round(mean_id, 4),
        "typographic_H":    round(mean_ty, 4),
        "excluded":         flag,
    })

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# Also run for encoder models using CUI predictions from RQ1
for model_name in df_enc_rq2["model_name"].unique():
    mdf = df_enc_rq2[df_enc_rq2["model_name"]==model_name]
    # Encoder identical control: entropy from RQ1 where n_unique_preds == 1
    # (proxy: instances where all perts mapped to same CUI)
    stable_frac = (
        df_encoder[
            (df_encoder["model_name"]==model_name) &
            (df_encoder["input_type"]=="perturbation")
        ].groupby("instance_id")["predicted_cui_or_cluster"]
        .nunique()
    )
    mean_id_enc = (stable_frac == 1).mean()
    nc_rows.append({
        "model_name":       model_name,
        "architecture":     "encoder",
        "identical_copy_H": round(1.0 - mean_id_enc, 4),
        "typographic_H":    float("nan"),
        "excluded":         False,
    })

df_nc = pd.DataFrame(nc_rows)
print("\nNegative control summary:")
print(df_nc.to_string(index=False))
df_nc.to_csv(TABLES_DIR / "rq2_negative_controls.csv", index=False)

# Remove excluded models from df_all
if EXCLUDED_MODELS_RQ2:
    df_all = df_all[
        ~df_all["model_name"].isin(EXCLUDED_MODELS_RQ2)
    ].copy()
    print(f"\nRows after exclusion: {len(df_all)}")

## 7) Pre-Registered Statistical Analysis

**Primary analysis (SQ1):**
One-tailed Wilcoxon signed-rank test comparing H-hat in the top
accuracy quartile vs bottom accuracy quartile per model.
Pre-specified threshold: rank-biserial r >= 0.30

**Global dissociation test (SQ1 + SQ3):**
Spearman rank correlation between per-instance accuracy and H-hat,
pooled via Fisher z-transformation.
Pre-specified threshold: |rho| < 0.30 confirms dissociation.

**Architecture comparison (SQ2):**
Compare dissociation strength between encoder-only and generative
models using the rank-biserial effect sizes.

**Corrections:**
BH-FDR at q = 0.05 across all confirmatory tests.

In [ ]:
from scipy.stats import wilcoxon, spearmanr, rankdata
import numpy as np

def rank_biserial_r(x, y):
    """
    Rank-biserial correlation for Wilcoxon signed-rank test.
    r = 1 - (2 * W) / (n * (n+1) / 2)
    where W is the smaller rank sum.
    Pre-specified threshold: r >= 0.30 for meaningful effect.
    """
    n = len(x)
    if n == 0:
        return np.nan
    diffs = np.array(y) - np.array(x)
    diffs = diffs[diffs != 0]
    n = len(diffs)
    if n == 0:
        return 0.0
    ranks = rankdata(np.abs(diffs))
    W_pos = ranks[diffs > 0].sum()
    W_neg = ranks[diffs < 0].sum()
    W     = min(W_pos, W_neg)
    r     = 1 - (2 * W) / (n * (n + 1) / 2)
    return float(r)


def bh_fdr(pvals):
    pvals = np.array(pvals, dtype=float)
    n     = len(pvals)
    idx   = np.argsort(pvals)
    bh    = pvals[idx] * n / np.arange(1, n + 1)
    for i in range(n - 2, -1, -1):
        bh[idx[i]] = min(bh[idx[i]], bh[idx[i + 1]])
    return np.minimum(bh, 1.0)


stat_rows = []
models    = df_all["model_name"].unique()

for model_name in models:
    mdf  = df_all[df_all["model_name"] == model_name].dropna(
        subset=["normalised_entropy", "mean_accuracy"]
    ).copy()
    arch = mdf["architecture"].iloc[0]

    # Quartile split
    q_top = mdf["mean_accuracy"].quantile(TOP_Q)
    q_bot = mdf["mean_accuracy"].quantile(BOT_Q)
    top   = mdf[mdf["mean_accuracy"] >= q_top]["normalised_entropy"].values
    bot   = mdf[mdf["mean_accuracy"] <= q_bot]["normalised_entropy"].values

    # Wilcoxon signed-rank (one-tailed: top > bot)
    n_pairs = min(len(top), len(bot))
    if n_pairs >= 5:
        top_s = np.random.default_rng(SEED).choice(top, size=n_pairs, replace=False)
        bot_s = np.random.default_rng(SEED).choice(bot, size=n_pairs, replace=False)
        try:
            stat_w, p_w = wilcoxon(top_s, bot_s, alternative="greater")
            rb_r        = rank_biserial_r(bot_s, top_s)
        except Exception as e:
            stat_w, p_w, rb_r = np.nan, np.nan, np.nan
            print(f"  [WARN] Wilcoxon failed for {model_name}: {e}")
    else:
        stat_w, p_w, rb_r = np.nan, np.nan, np.nan

    # Spearman rho (global dissociation)
    rho, p_rho = spearmanr(
        mdf["mean_accuracy"], mdf["normalised_entropy"]
    ) if len(mdf) >= 5 else (np.nan, np.nan)

    # Mean entropy in top vs bottom quartile
    mean_top = top.mean() if len(top) > 0 else np.nan
    mean_bot = bot.mean() if len(bot) > 0 else np.nan

    stat_rows.append({
        "model_name":     model_name,
        "architecture":   arch,
        "n_instances":    len(mdf),
        "mean_entropy":   mdf["normalised_entropy"].mean(),
        "mean_accuracy":  mdf["mean_accuracy"].mean(),
        "top_q_mean_H":   mean_top,
        "bot_q_mean_H":   mean_bot,
        "wilcoxon_stat":  stat_w,
        "wilcoxon_p":     p_w,
        "rank_biserial_r":rb_r,
        "meets_rb_threshold": (rb_r >= 0.30) if not np.isnan(rb_r) else False,
        "spearman_rho":   rho,
        "spearman_p":     p_rho,
        "dissociation_confirmed": (abs(rho) < 0.30) if not np.isnan(rho) else False,
    })

df_stats = pd.DataFrame(stat_rows)

# BH-FDR across all Wilcoxon tests
pvals_w = df_stats["wilcoxon_p"].fillna(1.0).tolist()
df_stats["wilcoxon_p_bh"] = bh_fdr(pvals_w)
df_stats["wilcoxon_sig"]  = df_stats["wilcoxon_p_bh"] < 0.05

print("=" * 70)
print("RQ2 STATISTICAL RESULTS")
print("=" * 70)
print(df_stats[[
    "model_name", "architecture",
    "top_q_mean_H", "bot_q_mean_H",
    "rank_biserial_r", "meets_rb_threshold",
    "wilcoxon_p_bh", "wilcoxon_sig",
    "spearman_rho", "dissociation_confirmed",
]].to_string(index=False))

# ── Fisher z-transformation: pool Spearman rho across models ─────────────
# Pre-registered: pool per-model rho values using Fisher z to get
# a single pooled correlation estimate with 95% CI.
# This tests the global accuracy-stability dissociation claim.
def fisher_z_pool(rhos, ns):
    """Pool Spearman correlations via Fisher z-transformation."""
    rhos = np.array([r for r in rhos if not np.isnan(r)])
    ns   = np.array([n for n, r in zip(ns, rhos)
                     if not np.isnan(r)])
    if len(rhos) == 0:
        return np.nan, np.nan, np.nan
    z_vals  = np.arctanh(rhos)
    weights = ns - 3
    z_pool  = np.average(z_vals, weights=weights)
    se_pool = 1 / np.sqrt(weights.sum())
    rho_pool = np.tanh(z_pool)
    ci_low   = np.tanh(z_pool - 1.96 * se_pool)
    ci_high  = np.tanh(z_pool + 1.96 * se_pool)
    return float(rho_pool), float(ci_low), float(ci_high)

_rhos_all = df_stats["spearman_rho"].tolist()
_ns_all   = df_stats["n_instances"].tolist()
_rho_p, _ci_lo, _ci_hi = fisher_z_pool(_rhos_all, _ns_all)

# Separate by architecture
_enc = df_stats[df_stats["architecture"]=="encoder"]
_gen = df_stats[df_stats["architecture"]=="generative"]
_rho_enc, _lo_enc, _hi_enc = fisher_z_pool(
    _enc["spearman_rho"].tolist(), _enc["n_instances"].tolist())
_rho_gen, _lo_gen, _hi_gen = fisher_z_pool(
    _gen["spearman_rho"].tolist(), _gen["n_instances"].tolist())

print("\n── Fisher z-pooled Spearman ρ (Global Dissociation Test) ──")
print(f"  All models:     ρ_pooled={_rho_p:.4f}  "
      f"95% CI [{_ci_lo:.4f}, {_ci_hi:.4f}]")
print(f"  Encoder only:   ρ_pooled={_rho_enc:.4f}  "
      f"95% CI [{_lo_enc:.4f}, {_hi_enc:.4f}]")
print(f"  Generative only:ρ_pooled={_rho_gen:.4f}  "
      f"95% CI [{_lo_gen:.4f}, {_hi_gen:.4f}]")
print(f"\n  Pre-registered threshold: |ρ_pooled| < 0.30")
print(f"  Global dissociation confirmed: {abs(_rho_p) < 0.30}")

df_stats["fisher_z_rho_pooled"]  = _rho_p
df_stats["fisher_z_ci_low"]      = _ci_lo
df_stats["fisher_z_ci_high"]     = _ci_hi

df_stats.to_csv(TABLES_DIR / "rq2_statistics.csv", index=False)

## 8) Figures

Four figures for RQ2:
- Figure 1: Accuracy vs Entropy scatter (all models)
- Figure 2: Top vs Bottom quartile entropy comparison
- Figure 3: Encoder vs Generative architecture comparison
- Figure 4: Per-model dissociation summary bar chart

### Figure 1 - Accuracy vs Entropy Scatter
*Caption:* Scatter plot of mean task accuracy (x-axis) vs
normalised semantic entropy H-hat (y-axis) for each model.
Green shading marks the top accuracy quartile - the key region
for RQ2 SQ1. If dissociation exists, high-accuracy instances
should show elevated entropy, not near-zero entropy.
Spearman ρ shown in each panel title.

In [ ]:
fig1, axes = plt.subplots(2, 3, figsize=(16, 10), sharey=True, sharex=True)
axes = axes.flatten()
arch_colors = {"encoder": "steelblue", "generative": "darkorange"}

for ax, model_name in zip(axes, df_all["model_name"].unique()):
    mdf  = df_all[df_all["model_name"] == model_name]
    arch = mdf["architecture"].iloc[0]
    col  = arch_colors[arch]

    ax.scatter(mdf["mean_accuracy"], mdf["normalised_entropy"],
               alpha=0.35, s=18, color=col)

    rho_val = df_stats.loc[
        df_stats["model_name"] == model_name, "spearman_rho"
    ].values
    rho_str = f"ρ={rho_val[0]:.3f}" if len(rho_val) > 0 else ""
    ax.set_title(f"{model_name}\n({arch}) {rho_str}",
                 fontsize=9, fontweight="bold")
    ax.set_xlabel("Mean Accuracy", fontsize=8)
    ax.set_ylabel("Normalised Entropy Ĥ", fontsize=8)
    ax.axvline(0.5, ls="--", color="grey", lw=0.8, alpha=0.5)

    # Shade top-accuracy quartile
    q_top = mdf["mean_accuracy"].quantile(TOP_Q)
    ax.axvspan(q_top, mdf["mean_accuracy"].max(),
               alpha=0.08, color="green",
               label="Top accuracy quartile")

# Hide unused subplots
for ax in axes[len(df_all["model_name"].unique()):]:
    ax.set_visible(False)

fig1.suptitle(
    "RQ2 Figure 1. Accuracy vs Semantic Entropy per Model\n"
    "(Green shading = top accuracy quartile; RQ2 tests whether "
    "entropy remains high here)",
    fontsize=11, fontweight="bold")
plt.tight_layout()
fig1.savefig(FIGURES_DIR / "rq2_figure1_accuracy_vs_entropy.png",
             dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rq2_figure1_accuracy_vs_entropy.png")

### Figure 2 - Top vs Bottom Accuracy Quartile Entropy
*Caption:* Mean H-hat in the top 25% accuracy quartile (green)
vs bottom 25% (red) for each model. Pre-registered test: if
accuracy-stability dissociation exists, the bars should be
similar height - high accuracy does NOT imply low entropy.

In [ ]:
_plot_rows = []
for _, r in df_stats.iterrows():
    _plot_rows.append({
        "model_name": r["model_name"],
        "architecture": r["architecture"],
        "quartile": "Top 25% accuracy",
        "mean_entropy": r["top_q_mean_H"],
    })
    _plot_rows.append({
        "model_name": r["model_name"],
        "architecture": r["architecture"],
        "quartile": "Bottom 25% accuracy",
        "mean_entropy": r["bot_q_mean_H"],
    })
_qdf = pd.DataFrame(_plot_rows).dropna()

fig2, ax2 = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=_qdf,
    x="model_name", y="mean_entropy",
    hue="quartile",
    palette={"Top 25% accuracy": "#2e7d32",
             "Bottom 25% accuracy": "#c62828"},
    ax=ax2,
)
ax2.axhline(0.30, ls="--", color="navy", lw=1.2, alpha=0.7,
            label="Pre-registered r=0.30 reference")
ax2.set_title(
    "RQ2 Figure 2. Mean Entropy: Top vs Bottom Accuracy Quartile\n"
    "(If dissociation exists: top quartile entropy ≈ bottom quartile entropy)",
    fontsize=11, fontweight="bold")
ax2.set_xlabel("Model", fontsize=11)
ax2.set_ylabel("Mean Normalised Entropy Ĥ", fontsize=11)
ax2.set_ylim(0, 1.0)
plt.xticks(rotation=20, ha="right")
plt.legend()
plt.tight_layout()
fig2.savefig(FIGURES_DIR / "rq2_figure2_quartile_comparison.png",
             dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rq2_figure2_quartile_comparison.png")

### Figure 3 - Encoder vs Generative Architecture Comparison
*Caption:* Boxplot comparing entropy distributions across
encoder (CUI-ranking) and generative (text-generation)
architectures. Addresses SQ2 - is the dissociation pattern
architecture-specific or general?

In [ ]:
fig3, ax3 = plt.subplots(figsize=(10, 6))

# Compute mean and std per architecture
_arch_stats = df_all.groupby("architecture")[
    "normalised_entropy"
].agg(["mean","std","count"]).reset_index()
_arch_stats["se"]   = _arch_stats["std"] / np.sqrt(_arch_stats["count"])
_arch_stats["ci95"] = 1.96 * _arch_stats["se"]

_colors = {"encoder": "steelblue", "generative": "darkorange"}
_x      = np.arange(len(_arch_stats))

ax3.bar(_x, _arch_stats["mean"],
        yerr=_arch_stats["ci95"],
        color=[_colors[a] for a in _arch_stats["architecture"]],
        alpha=0.75, width=0.4, capsize=8,
        error_kw={"elinewidth":2, "ecolor":"black"})

# Overlay individual model points
for _, row in df_all.groupby(["architecture","model_name"])[
        "normalised_entropy"].mean().reset_index().iterrows():
    _xi = 0 if row["architecture"]=="encoder" else 1
    ax3.scatter(_xi, row["normalised_entropy"],
                color=_colors[row["architecture"]],
                edgecolors="black", s=80, zorder=5, alpha=0.8)
    ax3.text(_xi + 0.05, row["normalised_entropy"],
             row["model_name"], fontsize=7.5,
             va="center", color="dimgrey")

ax3.set_xticks(_x)
ax3.set_xticklabels(
    [f"{a.capitalize()}\n(n={int(r.count)})"
     for a, r in zip(_arch_stats["architecture"],
                     _arch_stats.itertuples())],
    fontsize=11)
ax3.set_xlabel("Architecture", fontsize=11)
ax3.set_ylabel("Normalised Entropy Ĥ", fontsize=11)
ax3.set_ylim(0, 1.0)
ax3.set_title(
    "RQ2 Figure 3. Entropy Distribution: Encoder vs Generative\n"
    "(Addresses SQ2 - is dissociation architecture-specific?)\n"
    "Bars = mean ± 95% CI; dots = individual model means",
    fontsize=11, fontweight="bold")
sns.despine()
plt.tight_layout()
fig3.savefig(FIGURES_DIR / "rq2_figure3_architecture_comparison.png",
             dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rq2_figure3_architecture_comparison.png")

### Figure 4 - Dissociation Evidence Summary
*Caption:* Left: rank-biserial r per model (pre-registered
threshold ≥ 0.30 shown as dashed line). Right: Spearman |ρ|
per model (dissociation confirmed if |ρ| < 0.30, shown as
dashed line). Green bars = threshold met. Red bars = not met.

In [ ]:
fig4, axes4 = plt.subplots(1, 2, figsize=(14, 6))

# Left: rank-biserial r per model
colors_bar = ["#2e7d32" if v else "#c62828"
              for v in df_stats["meets_rb_threshold"]]
axes4[0].bar(df_stats["model_name"], df_stats["rank_biserial_r"],
             color=colors_bar, alpha=0.8)
axes4[0].axhline(0.30, ls="--", color="navy", lw=1.5,
                 label="Pre-registered threshold r=0.30")
axes4[0].set_title("Rank-biserial r\n(Top vs Bottom Accuracy Quartile)",
                   fontsize=10, fontweight="bold")
axes4[0].set_ylabel("Rank-biserial r", fontsize=10)
axes4[0].set_ylim(0, 1.0)
axes4[0].set_xticklabels(df_stats["model_name"], rotation=20, ha="right")
axes4[0].legend()

# Right: Spearman rho per model
colors_rho = ["#2e7d32" if v else "#c62828"
              for v in df_stats["dissociation_confirmed"]]
axes4[1].bar(df_stats["model_name"],
             df_stats["spearman_rho"].abs(),
             color=colors_rho, alpha=0.8)
axes4[1].axhline(0.30, ls="--", color="navy", lw=1.5,
                 label="Pre-registered threshold |ρ|<0.30")
axes4[1].set_title("Spearman |ρ| (Accuracy vs Entropy)\n"
                   "(Green = dissociation confirmed: |ρ| < 0.30)",
                   fontsize=10, fontweight="bold")
axes4[1].set_ylabel("|Spearman ρ|", fontsize=10)
axes4[1].set_ylim(0, 1.0)
axes4[1].set_xticklabels(df_stats["model_name"], rotation=20, ha="right")
axes4[1].legend()

fig4.suptitle("RQ2 Figure 4. Dissociation Evidence per Model",
              fontsize=12, fontweight="bold")
plt.tight_layout()
fig4.savefig(FIGURES_DIR / "rq2_figure4_dissociation_summary.png",
             dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rq2_figure4_dissociation_summary.png")

## 9) RQ2 Conclusions - Sub-question Answers

In [ ]:
SEP = "=" * 70
print(SEP)
print("RQ2 CONCLUSIONS")
print(SEP)

enc_stats = df_stats[df_stats["architecture"] == "encoder"]
gen_stats = df_stats[df_stats["architecture"] == "generative"]

n_dissoc_enc = enc_stats["dissociation_confirmed"].sum()
n_dissoc_gen = gen_stats["dissociation_confirmed"].sum()
n_rb_enc     = enc_stats["meets_rb_threshold"].sum()
n_rb_gen     = gen_stats["meets_rb_threshold"].sum()

print()
print("SQ1 - Can high-accuracy models still produce high entropy?")
print(f"  Encoder models with dissociation confirmed (|rho|<0.30): "
      f"{n_dissoc_enc}/{len(enc_stats)}")
print(f"  Generative models with dissociation confirmed:           "
      f"{n_dissoc_gen}/{len(gen_stats)}")
print()
print("SQ2 - Is dissociation architecture-specific?")
print(f"  Encoder models meeting rank-biserial r>=0.30:   {n_rb_enc}/{len(enc_stats)}")
print(f"  Generative models meeting rank-biserial r>=0.30:{n_rb_gen}/{len(gen_stats)}")
print()
print("SQ3 - How often are apparently correct outputs semantically unstable?")
for _, r in df_stats.iterrows():
    mdf = df_all[df_all["model_name"] == r["model_name"]]
    high_acc_high_H = mdf[
        (mdf["mean_accuracy"] >= mdf["mean_accuracy"].quantile(TOP_Q)) &
        (mdf["normalised_entropy"] > 0.20)
    ]
    pct = 100 * len(high_acc_high_H) / max(len(mdf), 1)
    print(f"  {r['model_name']} ({r['architecture']}): "
          f"{pct:.1f}% of high-accuracy instances have H > 0.20")

print()
print(SEP)
print("Full statistical table:")
print(df_stats[["model_name","architecture",
                "rank_biserial_r","meets_rb_threshold",
                "spearman_rho","dissociation_confirmed",
                "wilcoxon_p_bh","wilcoxon_sig"]].to_string(index=False))
print(SEP)

# Save summary
summary = {
    "n_encoder_dissociation":   int(n_dissoc_enc),
    "n_generative_dissociation":int(n_dissoc_gen),
    "n_encoder_rb_met":         int(n_rb_enc),
    "n_generative_rb_met":      int(n_rb_gen),
    "models": df_stats[[
        "model_name","architecture",
        "rank_biserial_r","spearman_rho",
        "dissociation_confirmed","wilcoxon_sig"
    ]].to_dict("records"),
}
(OUTPUT_DIR / "rq2_summary.json").write_text(
    json.dumps(summary, indent=2, default=str))
print(f"\nSummary saved: {OUTPUT_DIR}/rq2_summary.json")

## 10) Reproducibility Metadata

In [ ]:
import datetime
metadata = {
    "rq":              "RQ2",
    "timestamp_utc":   datetime.datetime.utcnow().isoformat(),
    "seed":            SEED,
    "pilot_n":         PILOT_N,
    "k_perturb":       K_PERTURB,
    "bootstrap_b":     BOOTSTRAP_B,
    "use_int8":        USE_INT8,
    "do_sample":       DO_SAMPLE,
    "temperature":     TEMPERATURE,
    "top_quartile":    TOP_Q,
    "bot_quartile":    BOT_Q,
    "encoder_models":  ["BERT-base","BioBERT","PubMedBERT"],
    "generative_models": list(GEN_MODEL_SPECS.keys()),
    "gen_model_ids":   {k: v["hf_id"] for k, v in GEN_MODEL_SPECS.items()},
    "rq1_inter_path":  str(RQ1_INTER),
    "output_dir":      str(OUTPUT_DIR),
    "pre_reg_thresholds": {
        "rank_biserial_r": 0.30,
        "spearman_rho_max": 0.30,
        "bh_fdr_q": 0.05,
    },
}
(OUTPUT_DIR / "rq2_run_metadata.json").write_text(
    json.dumps(metadata, indent=2))
print("Metadata saved.")
print(json.dumps(metadata, indent=2))

## 11) Final Diagnostics

In [ ]:
# ── Final diagnostics ─────────────────────────────────────────────────────
print("=" * 60)
print("RQ2 FINAL DIAGNOSTICS")
print("=" * 60)
print(f"Encoder models used:    {sorted(df_enc_rq2['model_name'].unique().tolist())}")
print(f"Generative models used: {sorted(df_gen_entropy['model_name'].unique().tolist())}")
print(f"Excluded models:        {sorted(EXCLUDED_MODELS_RQ2) if EXCLUDED_MODELS_RQ2 else 'None'}")
print(f"Total instances (N):    {df_all['instance_id'].nunique()}")
print(f"Fisher z pooled rho:    {_rho_p:.4f}  (|rho|<0.30 = {abs(_rho_p)<0.30})")
n_diss = (df_stats["dissociation_confirmed"]==True).sum()
n_rb   = (df_stats["meets_rb_threshold"]==True).sum()
print(f"Models with dissociation confirmed: {n_diss}/{len(df_stats)}")
print(f"Models meeting r>=0.30:             {n_rb}/{len(df_stats)}")
print(f"All required outputs saved:         True")
print("=" * 60)